# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pylops
import time as _time

from scipy.ndimage        import shift   as nd_shift
from scipy.signal         import hilbert as sp_hilbert
from scipy.signal.windows import tukey

# Functions

In [ ]:
# -- Vectorised point-scatterer B-scan ----------------------------------------
def point_bscan(x_sc, z_sc, x_rec, t_arr, v, fc):
    r   = np.sqrt((x_rec - x_sc)**2 + z_sc**2)
    twt = 2.0 * r / v
    amp = 1.0 / np.sqrt(r)
    u   = np.pi * fc * (t_arr[:, None] - twt[None, :])
    return amp[None, :] * (1.0 - 2*u**2) * np.exp(-u**2)

# -- Z-direction Riesz transform ----------------------------------------------
def riesz_z(W):
    nz, nx = W.shape
    KX, KZ = np.meshgrid(np.fft.fftfreq(nx), np.fft.fftfreq(nz))
    K = np.sqrt(KX**2 + KZ**2);  K[0, 0] = 1.0
    return np.real(np.fft.ifft2((-1j * KZ / K) * np.fft.fft2(W)))

In [ ]:

def gazdag_migration(data, x, t, z, vel):
    """
    Gazdag (1978) phase-shift migration for zero-offset post-stack data.

    data : (n_t, n_x)  B-scan, time axis first, t0-shifted
    x, t : 1-D arrays [m, ns]
    z    : 1-D depth axis [m], uniformly spaced from 0
    vel  : full medium velocity [m/ns]; v_mig = vel/2 used internally

    Returns image (n_z, n_x), peak-normalised to +-1.
    """
    import numpy as np
    n_t, n_x = data.shape
    dt    = float(t[1] - t[0])
    dx    = float(x[1] - x[0])
    dz    = float(z[1] - z[0])
    v_mig = vel / 2.0

    # Spatial cosine taper (5%) + 100% zero-pad each side
    n_xtap  = max(3, int(0.05 * n_x))
    pad     = n_x
    sp_tap  = pylops.utils.tapers.taper2d(n_t, n_x, n_xtap)   # (n_x, n_t)
    data_sp = np.pad(
        data.T * sp_tap,            # (n_x, n_t)
        ((pad, pad), (0, 0)),
        mode='constant'
    )                               # (nx_pad, n_t)
    nx_pad = n_x + 2 * pad

    # f-kx evanescent filter
    D_fk       = np.fft.fft(np.fft.rfft(data_sp, axis=1), axis=0)
    freq_arr   = np.fft.rfftfreq(n_t, dt)
    kx_arr     = np.fft.fftfreq(nx_pad, dx)
    evanescent = np.abs(kx_arr[:, None]) > np.abs(freq_arr[None, :]) / v_mig
    D_fk[evanescent] = 0.0
    data_sp = np.real(np.fft.irfft(np.fft.ifft(D_fk, axis=0), n=n_t, axis=1))

    print(f'  nx={n_x} -> nx_pad={nx_pad},  evanescent: '
          f'{evanescent.sum()}/{evanescent.size} ({100*evanescent.mean():.1f}%)')

    # PhaseShift depth loop: downward continuation + t=0 imaging condition
    freq  = np.fft.rfftfreq(n_t, dt)
    kx    = np.fft.fftshift(np.fft.fftfreq(nx_pad, dx))
    Pop   = pylops.waveeqprocessing.PhaseShift(v_mig, dz, n_t, freq, kx)
    field = data_sp.T.ravel()

    image = np.zeros((len(z), n_x))
    for iz in range(len(z)):
        image[iz] = np.real(field.reshape(n_t, nx_pad)[0, pad:-pad])
        if iz < len(z) - 1:
            field = Pop.H * field

    peak = np.max(np.abs(image))
    return image / (peak + 1e-30)


In [ ]:


def estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent):
    """
    Fit a 2D phase plane phi(kz,kx) = kz*dz + kx*dx + c to the cross-spectrum.
    
    Updates:
    - Uses a Tukey window to preserve core wavelet amplitudes.
    - Dynamically bounds the fitting to where the cross-spectrum energy is 
      strong, preventing phase-wrapping errors safely.
    """
    Nz, Nx = base.shape
    
    # 1. Compute 2D wavenumber axes
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    # 2. Apply Tukey window (tapers outer 15%, leaves inner 85% pristine)
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    
    # 3. Compute Cross-Spectrum
    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon * taper)
    XS       = base_fft * np.conj(mon_fft)

    # 4. Extract weights (magnitude) and phase
    w   = np.abs(XS)
    phi = np.angle(XS)

    # 5. Dynamic Band Limit Mask
    # Limits evaluation to a stable band around center frequencies to avoid 
    # phase wrapping, while filtering out low-energy noise floors (<10% peak)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    # 6. Linear Regression (Weighted Least Squares)
    A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
    W = w[mask]  # Amplitude weights ensure reliable bins dominate the fit
    
    # Solve system: (A * W) * c = (phi * W)
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    
    return c[0], c[1], XS, kz_ax, kx_ax

# Vertical, Horizontal, Diagonal Movement Test

# Tertiary variables

In [ ]:
# -- Acquisition --------------------------------------------------------------
dx_b   = 0.005 / 4
n_x_b  = 160 * 4
x_b    = np.arange(n_x_b) * dx_b       # 0 -- 0.795 m profile
x_sc0  = x_b.mean()

dt_b   = 0.004
n_t_b  = 1300                           # t_max = 5.196 ns > TWT(0.30 m) = 3.57 ns
t_b    = np.arange(n_t_b) * dt_b

z_sc   = 0.30
z_mig  = np.arange(0, 0.42, dx_b)      # 84 depth steps

In [ ]:
v_ice  = 0.168        # m/ns
f_c    = 1.5          # GHz
lam    = v_ice / f_c  # wavelength in ice [m]

In [ ]:
# =============================================================================
# Vertical Shift — Phase Informed Timelapse Migration
#
# Scatterer moves downward by dz_shift = shift_frac_v * lam.
# =============================================================================

# ── Parameters ───────────────────────────────────────────────────────────────
# v_ice, f_c, lam, x_b, t_b, z_mig, dx_b, x_sc0, z_sc  from earlier cells
dz_b = float(z_mig[1] - z_mig[0])

shift_frac_v = 1/8          # vertical shift as fraction of wavelength  <-- tune
dz_shift     = shift_frac_v * lam
z_sc_mon_v   = z_sc + dz_shift   # monitor scatterer depth

print(f"lambda      = {lam*1e3:.2f} mm")
print(f"vert shift  = {shift_frac_v} x lambda = {dz_shift*1e3:.2f} mm  ({dz_shift/dz_b:.1f} samples)")
print(f"Baseline:   x = {x_sc0*1e3:.1f} mm,  z = {z_sc*1e3:.1f} mm")
print(f"Monitor:    x = {x_sc0*1e3:.1f} mm,  z = {z_sc_mon_v*1e3:.1f} mm")

# ── 1. Synthesise B-scans ─────────────────────────────────────────────────────
print("\nSynthesising B-scans...")
B_v_base = point_bscan(x_sc0, z_sc,       x_b, t_b, v_ice, f_c)
B_v_mon  = point_bscan(x_sc0, z_sc_mon_v, x_b, t_b, v_ice, f_c)

# ── 2. Migrate both ───────────────────────────────────────────────────────────
print("\nMigrating baseline...")
t0 = _time.time()
mig_v_base = gazdag_migration(B_v_base, x_b, t_b, z_mig, v_ice)
print(f"  done in {_time.time()-t0:.1f} s")

print("\nMigrating monitor...")
t0 = _time.time()
mig_v_mon  = gazdag_migration(B_v_mon,  x_b, t_b, z_mig, v_ice)
print(f"  done in {_time.time()-t0:.1f} s")

# ── 3. Migrated difference ────────────────────────────────────────────────────
mig_v_diff = mig_v_base - mig_v_mon

# ── 4. Instantaneous amplitude & phase (Riesz-z) ─────────────────────────────
R_v_base = riesz_z(mig_v_base)
R_v_mon  = riesz_z(mig_v_mon)

A_v_base = np.sqrt(mig_v_base**2 + R_v_base**2)
A_v_mon  = np.sqrt(mig_v_mon**2  + R_v_mon**2)

phi_v_base  = np.arctan2(R_v_base, mig_v_base)
phi_v_mon   = np.arctan2(R_v_mon,  mig_v_mon)
phi_v_delta = np.angle(np.exp(1j * (phi_v_mon - phi_v_base)))

# ── 5. Amplitude-masked phase images ─────────────────────────────────────────
thresh_frac_v = 0.08

mask_v_base  = A_v_base < thresh_frac_v * A_v_base.max()
mask_v_mon   = A_v_mon  < thresh_frac_v * A_v_mon.max()
mask_v_delta = mask_v_base | mask_v_mon

phi_v_base_msk  = np.where(mask_v_base,  np.nan, phi_v_base)
phi_v_mon_msk   = np.where(mask_v_mon,   np.nan, phi_v_mon)
phi_v_delta_msk = np.where(mask_v_delta, np.nan, phi_v_delta)

print(f"\nAmplitude mask ({thresh_frac_v*100:.0f}% of peak):")
print(f"  baseline: {mask_v_base.mean()*100:.1f}%  monitor: {mask_v_mon.mean()*100:.1f}%  delta: {mask_v_delta.mean()*100:.1f}%")

# ── 6. QC display (4 x 3) ────────────────────────────────────────────────────
ext_b   = [x_b[0], x_b[-1], t_b[-1], t_b[0]]
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]

x_mid_v = x_sc0
z_mid_v = 0.5 * (z_sc + z_sc_mon_v)
x_lo_v  = x_mid_v - 5 * lam;  x_hi_v = x_mid_v + 5 * lam
z_lo_v  = z_mid_v - 0.10;     z_hi_v = z_mid_v + 0.10

def mig_zoom_v(ax):
    ax.set_xlim(x_lo_v, x_hi_v);  ax.set_ylim(z_hi_v, z_lo_v)
    ax.plot(x_sc0, z_sc,       'g*', ms=12, zorder=7, label='baseline')
    ax.plot(x_sc0, z_sc_mon_v, 'r^', ms=10, zorder=7, label=f'monitor ({shift_frac_v}\u03bb)')

clip_b_v   = np.percentile(np.abs(B_v_base), 99)
clip_diff_v = np.percentile(np.abs(mig_v_diff), 99)
hsv_cmap_v  = plt.get_cmap('hsv').copy();  hsv_cmap_v.set_bad('lightgrey')

fig, axes = plt.subplots(4, 3, figsize=(18, 18))
fig.suptitle(
    f"Vertical Shift — Phase Informed Timelapse Migration\n"
    f"shift = {shift_frac_v}\u03bb = {dz_shift*1e3:.1f} mm downward",
    fontsize=13, fontweight='bold',
)

# Row 0: B-scans
for ax, data, title in [
    (axes[0,0], B_v_base, "Baseline B-scan"),
    (axes[0,1], B_v_mon,  f"Monitor B-scan (shift {shift_frac_v}\u03bb down)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_b,
                   cmap='RdBu', vmin=-clip_b_v, vmax=clip_b_v, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("TWT [ns]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

B_v_diff = B_v_mon - B_v_base
im = axes[0,2].imshow(B_v_diff, aspect='auto', origin='upper', extent=ext_b,
                       cmap='RdBu', vmin=-np.percentile(np.abs(B_v_diff),99),
                       vmax=np.percentile(np.abs(B_v_diff),99), interpolation='bilinear')
axes[0,2].set_title("Difference B-scan", fontsize=10)
axes[0,2].set_xlabel("Position [m]");  axes[0,2].set_ylabel("TWT [ns]")
fig.colorbar(im, ax=axes[0,2], fraction=0.046, pad=0.04)

# Row 1: Migrated images
for ax, data, clip, title in [
    (axes[1,0], mig_v_base, 1.0,         "Migrated baseline"),
    (axes[1,1], mig_v_mon,  1.0,         f"Migrated monitor ({shift_frac_v}\u03bb down)"),
    (axes[1,2], mig_v_diff, clip_diff_v, "Migrated difference (baseline - monitor)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom_v(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 2: Unmasked instantaneous phase
for ax, data, title in [
    (axes[2,0], phi_v_base,  r"Inst. phase $\varphi_{base}$"),
    (axes[2,1], phi_v_mon,   r"Inst. phase $\varphi_{mon}$"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom_v(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

im = axes[2,2].imshow(phi_v_delta, aspect='auto', origin='upper', extent=ext_mig,
                      cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='bilinear')
axes[2,2].set_title(r"Phase diff $\Delta\varphi$ (wrapped)", fontsize=10)
axes[2,2].set_xlabel("Position [m]");  axes[2,2].set_ylabel("Depth [m]")
mig_zoom_v(axes[2,2]);  axes[2,2].legend(fontsize=8)
fig.colorbar(im, ax=axes[2,2], fraction=0.046, pad=0.04, label='[rad]')

# Row 3: Amplitude-masked phase
for ax, data, title in [
    (axes[3,0], phi_v_base_msk,  f"$\\varphi_{{base}}$ masked ({thresh_frac_v*100:.0f}%)"),
    (axes[3,1], phi_v_mon_msk,   f"$\\varphi_{{mon}}$ masked ({thresh_frac_v*100:.0f}%)"),
    (axes[3,2], phi_v_delta_msk, f"$\\Delta\\varphi$ masked (either < {thresh_frac_v*100:.0f}%)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap=hsv_cmap_v, vmin=-np.pi, vmax=np.pi, interpolation='none')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom_v(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

plt.tight_layout()
plt.show()


In [ ]:
dz_mig = z_mig[1] - z_mig[0]
dx_mig = x_b[1]   - x_b[0]

kz_c = 2 * np.pi / lam
sf   = 1/8

cases = [
    ("Vertical",   sf * lam,              0.0                  ),
    ("Horizontal", 0.0,                   sf * lam             ),
    ("Diagonal",   sf * lam / np.sqrt(2), sf * lam / np.sqrt(2)),
]

fig, axes = plt.subplots(3, 4, figsize=(20, 11))
fig.suptitle(
    f"2D cross-spectrum shift estimation  |  "
    f"||shift|| = lambda * {sf} = {sf * lam * 1e3:.1f} mm  (all cases)",
    fontsize=12
)

for row, (name, true_dz, true_dx) in enumerate(cases):

    mon = nd_shift(mig_v_base, (true_dz / dz_mig, true_dx / dx_mig), order=3)

    dz_est, dx_est, XS, kz_ax, kx_ax = estimate_shift_2d(
        mig_v_base, mon, dz_mig, dx_mig, kz_c
    )

    XS_s     = np.fft.fftshift(XS)
    kz_s     = np.fft.fftshift(kz_ax)
    kx_s     = np.fft.fftshift(kx_ax)
    energy   = np.abs(XS_s)
    phi_show = np.where(energy > 0.05 * energy.max(),
                        np.degrees(np.angle(XS_s)), np.nan)

    # Col 0: difference image
    ax   = axes[row, 0]
    diff = mon - mig_v_base
    vmax = np.max(np.abs(diff))
    ax.pcolormesh(x_b * 100, z_mig * 100, diff,
                  cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(
        f"{name}  (dz={true_dz*1e3:.1f} mm, dx={true_dx*1e3:.1f} mm)\n"
        "Difference image (mon - base)", fontsize=9
    )
    ax.set_xlabel("x [cm]");  ax.set_ylabel("z [cm]")
    ax.invert_yaxis()

    # Col 1: cross-spectrum phase
    ax2 = axes[row, 1]
    im2 = ax2.pcolormesh(kx_s, kz_s, phi_show,
                         cmap="RdBu_r", vmin=-180, vmax=180, shading="auto")
    klim = 1.5 * kz_c
    for sgn in [-1, 1]:
        ax2.axhline(sgn * klim, color="k", lw=0.8, ls="--", alpha=0.5)
        ax2.axvline(sgn * klim, color="k", lw=0.8, ls="--", alpha=0.5)
    ax2.set_title("Cross-spectrum phase [deg]\ndashed = fit band", fontsize=9)
    ax2.set_xlabel("kx [rad/m]");  ax2.set_ylabel("kz [rad/m]")
    ax2.set_xlim(kx_s.min() / 4, kx_s.max() / 4)
    ax2.set_ylim(kz_s.min() / 4, kz_s.max() / 4)
    plt.colorbar(im2, ax=ax2, fraction=0.046)

    # Col 2: energy |XS|
    ax4 = axes[row, 2]
    im4 = ax4.pcolormesh(kx_s, kz_s, energy,
                         cmap="inferno", shading="auto")
    for sgn in [-1, 1]:
        ax4.axhline(sgn * klim, color="w", lw=0.8, ls="--", alpha=0.5)
        ax4.axvline(sgn * klim, color="w", lw=0.8, ls="--", alpha=0.5)
    ax4.set_title("Cross-spectrum energy |XS|\ndashed = fit band", fontsize=9)
    ax4.set_xlabel("kx [rad/m]");  ax4.set_ylabel("kz [rad/m]")
    ax4.set_xlim(kx_s.min() / 4, kx_s.max() / 4)
    ax4.set_ylim(kz_s.min() / 4, kz_s.max() / 4)
    plt.colorbar(im4, ax=ax4, fraction=0.046)

    # Col 3: results
    ax3 = axes[row, 3]
    ax3.axis("off")
    txt = (
        f"True:  dz = {true_dz * 1e3:+7.3f} mm\n"
        f"       dx = {true_dx * 1e3:+7.3f} mm\n\n"
        f"Est:   dz = {dz_est  * 1e3:+7.3f} mm\n"
        f"       dx = {dx_est  * 1e3:+7.3f} mm\n\n"
        f"Err:   dz = {(dz_est - true_dz) * 1e3:+.5f} mm\n"
        f"       dx = {(dx_est - true_dx) * 1e3:+.5f} mm"
    )
    ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
             fontsize=10, family="monospace", verticalalignment="top")
    ax3.set_title("Results", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 2D cross-spectrum estimator: error vs shift magnitude
# Row 0: absolute error |est - true| [mm]
# Row 1: relative error |est - true| / ||true shift||  [%]
#         normalised by the total applied shift magnitude so that small and
#         large shifts are on a comparable footing.
# =============================================================================

 
shift_fracs = np.linspace(0.02, 0.60, 40)

case_defs = [
    ('Vertical',
     lambda s: (s * lam,              0.0              ),
     1.0 / 3.0,
     'steelblue', 'cornflowerblue'),
    ('Horizontal',
     lambda s: (0.0,                  s * lam          ),
     1.0 / 3.0,
     'tomato', 'lightsalmon'),
    ('Diagonal',
     lambda s: (s * lam / np.sqrt(2), s * lam / np.sqrt(2)),
     1.0 / (3.0 * np.sqrt(2)),
     'seagreen', 'mediumaquamarine'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey='row')
fig.suptitle(
    f'2D cross-spectrum: error vs shift magnitude  '
    f'(noiseless, band-limited to 1.5*kz_c)\n'
    f'lambda = {lam*1e3:.1f} mm  |  '
    f'dz_sample = {dz_mig*1e3:.2f} mm  |  '
    f'dx_sample = {dx_mig*1e3:.2f} mm',
    fontsize=10
)

for col, (name, shift_fn, wrap_lim, col_z, col_x) in enumerate(case_defs):
    ax_abs = axes[0, col]
    ax_rel = axes[1, col]

    err_z_abs, err_x_abs = [], []
    err_z_rel, err_x_rel = [], []

    for frac in shift_fracs:
        true_dz, true_dx = shift_fn(frac)
        norm_mm = frac * lam * 1e3          # total shift magnitude [mm]

        mon = nd_shift(mig_v_base,
                       (true_dz / dz_mig, true_dx / dx_mig), order=3)
        dz_est, dx_est, *_ = estimate_shift_2d(
            mig_v_base, mon, dz_mig, dx_mig, kz_c
        )

        ez = abs(dz_est - true_dz) * 1e3   # mm
        ex = abs(dx_est - true_dx) * 1e3

        err_z_abs.append(max(ez, 1e-5))
        err_x_abs.append(max(ex, 1e-5))
        err_z_rel.append(max(ez / norm_mm * 100, 1e-4))   # percent
        err_x_rel.append(max(ex / norm_mm * 100, 1e-4))

    err_z_abs = np.array(err_z_abs)
    err_x_abs = np.array(err_x_abs)
    err_z_rel = np.array(err_z_rel)
    err_x_rel = np.array(err_x_rel)

    for ax, y_z, y_x, ylabel, ref_lines in [
        (ax_abs, err_z_abs, err_x_abs,
         'Absolute error [mm]',
         [(dz_mig * 1e3,   'dimgray',    f'Sample spacing ({dz_mig*1e3:.2f} mm)'),
          (lam * 1e3 / 10, 'darkorange', f'lambda/10 ({lam*1e3/10:.1f} mm)'),
          (lam * 1e3 / 100,'goldenrod',  f'lambda/100 ({lam*1e3/100:.2f} mm)')]),
        (ax_rel, err_z_rel, err_x_rel,
         'Relative error [%]',
         [(100,   'dimgray',    '100 % (error = shift)'),
          (10,    'darkorange', '10 %'),
          (1,     'goldenrod',  '1 %'),
          (0.1,   'lightgreen', '0.1 %')]),
    ]:
        ax.semilogy(shift_fracs, y_z, '-o',  ms=4, lw=1.5,
                    color=col_z, label='|err dz|')
        ax.semilogy(shift_fracs, y_x, '--s', ms=4, lw=1.5,
                    color=col_x, label='|err dx|')

        for level, color, label in ref_lines:
            ax.axhline(level, color=color, ls=':', lw=1.1, alpha=0.8,
                       label=label)

        ax.axvline(wrap_lim, color='red', ls='--', lw=1.3,
                   label=f'Wrap limit ({wrap_lim:.2f} lambda)')

        ax.set_xlabel('Shift magnitude [lambda]')
        if col == 0:
            ax.set_ylabel(ylabel)
        ax.set_xlim(shift_fracs[0] - 0.01, 0.62)
        ax.legend(fontsize=7, loc='upper left')
        ax.grid(True, alpha=0.2, which='both')

    ax_abs.set_title(f'{name} shift')

plt.tight_layout()
plt.show()


# gprMax Data Test

In [ ]:
import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot

In [ ]:
# Medium
eps_r      = 3.15
v_ice      = 0.299792458 / np.sqrt(eps_r)   # m/ns ≈ 0.16892
f_c_GHz    = 1.5                              # centre frequency [GHz]
wavelength = v_ice / f_c_GHz                  # m ≈ 0.1126
v_mig      = v_ice / 2                        # exploding-reflector half-velocity [m/ns]
t0_ns      = np.sqrt(2) / f_c_GHz            # Ricker peak delay [ns] ≈ 0.943

# Survey geometry
domain_x   = 4.0
n_traces   = 380
trace_step = 0.01   # m
rx_offset  = 0.1    # m (source-receiver offset in .in file)
x_traces   = (0.1 + rx_offset / 2) + np.arange(n_traces) * trace_step   # midpoints [m]

# Scatterer geometry (shared across all datasets)
x_centre        = domain_x / 2
y_surface       = 0.9                          # air-ice interface [m from bottom]
y_scatterer     = y_surface - wavelength * 6  # cylinder centre [m from bottom]
z_scatterer     = y_surface - y_scatterer     # depth below surface [m] ≈ 0.676
radius_scat     = wavelength / 10             # radius of cylinder (point scatterer approximation) [m] ≈ 0.0028
z_top           = z_scatterer - radius_scat   # depth to cylinder top (first reflection)

separations = np.array([2, 1, 0.5, 0.25, 0.125, 0.0625]) * wavelength
x_s1_all    = np.round(x_centre - wavelength * 3 + separations, 3)
x_s2_all    = np.round(x_centre + wavelength * 3 - separations, 3)
labels      = ['Baseline', '2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ']

# Migration depth grid
z_img = np.linspace(0.0, 0.8, 160)
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

# # Paths
# STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\timelapse_study')

print(f'λ_ice       = {wavelength*1e3:.1f} mm')
print(f'v_ice       = {v_ice:.5f} m/ns,  v_mig = {v_mig:.5f} m/ns')
print(f't0_ns       = {t0_ns:.3f} ns')
print(f'z_scatterer = {z_scatterer:.3f} m  (centre),  z_top = {z_top:.4f} m')
print(f'x_traces    : {x_traces[0]:.3f} → {x_traces[-1]:.3f} m  ({n_traces} traces)')

In [ ]:
separation = np.array([0, 2, 1, 1/2, 1/4, 1/8, 1/16]) * wavelength  # m

x_centre = domain_x / 2
x_scatterer_1 = np.round(x_centre - wavelength * 3 + separation, 3) # 7 locations, scatterer that moves
x_scatterer_2 = np.round(x_centre + wavelength * 3, 3) # 7 locations, scatterer that is fixed

# check if discretisation is sufficient for CFL
dx_required = (v_ice / 4.5) / 10 # 4 GHz is the highest frequency component in the Ricker wavelet
if 0.002 < dx_required:
    print(f"Discretisation is sufficiently small")
if radius_scat < dx_required:
    print(f"Scatterer radius is smaller than discretisation")

In [ ]:
rxnumber    = 1
rxcomponent = 'Ez'

output_background, dt = get_output_data(
    'timelapse_study/background/background_merged.out', rxnumber, rxcomponent
)

output_baseline, _ = get_output_data(
    'timelapse_study/baseline/baseline_merged.out', rxnumber, rxcomponent
)

_paths = [
    'timelapse_study/baseline/baseline_merged.out',
    'timelapse_study/shift_2lambda/resolution_2lambda_merged.out',
    'timelapse_study/shift_1lambda/resolution_1lambda_merged.out',
    'timelapse_study/shift_0p5lambda/resolution_0p5lambda_merged.out',
    'timelapse_study/shift_0p25lambda/resolution_0p25lambda_merged.out',
    'timelapse_study/shift_0p125lambda/resolution_0p125lambda_merged.out',
    'timelapse_study/shift_0p0625lambda/resolution_0p0625lambda_merged.out',
]
raw_outputs      = [get_output_data(p, rxnumber, rxcomponent)[0] for p in _paths]
outputs_static   = [r - output_background for r in raw_outputs]      # background-subtracted

dt_ns   = dt * 1e9
n_t     = outputs_static[0].shape[0]
time_ns = np.arange(n_t) * dt_ns

# Convenience lists for the visualisation cells below
data_pre = [('Background', output_background)] + list(zip(labels, raw_outputs))
data_static = list(zip(labels, outputs_static))
labels_all = list(labels)           # all 7 labels (Baseline + 6 shifts)
labels = labels[1:]

print(f'dt = {dt_ns:.6f} ns,  n_t = {n_t},  t_max = {time_ns[-1]:.2f} ns')
print(f'Data shape (n_t × n_tr): {outputs_static[0].shape}  ({n_t} time samples × {n_traces} traces)')

In [ ]:
# ── Taper & t0-shift parameters ───────────────────────────────────────────────
taper_end_ns   = 14.0   # cosine ramp onset [ns]: zeros hyperbola tails beyond this time
taper_decay_ns = 1.0    # exponential decay time constant [ns]: de-emphasises late arrivals
ANGLE_AP       = 40     # Kirchhoff aperture half-angle [degrees]

# ── Functions ─────────────────────────────────────────────────────────────────
def make_end_taper(n_samples, taper_samples):
    """Unity everywhere, then half-cosine 1→0 over the last `taper_samples`."""
    win = np.ones(n_samples)
    if taper_samples > 0:
        ramp = 0.5 * (1 + np.cos(np.pi * np.arange(taper_samples) / taper_samples))
        win[-taper_samples:] = ramp
    return win

def make_decay_taper(time_ns, tau_ns):
    """Exponential decay starting at 1.0 (t=0) with time constant tau_ns."""
    return np.exp(-time_ns / tau_ns)

def preprocess(data_nt_ntr, dt_ns, t0_ns, time_ns):
    """
    Prepare a B-scan for Kirchhoff migration.
    Input:  data_nt_ntr  (n_t, n_tr)  — background-subtracted B-scan
    Returns tapered (n_tr, n_t) and t0-shifted (n_tr, n_t) arrays.
    """
    bscan   = data_nt_ntr.T.copy()                              # → (n_tr, n_t)
    n_t     = bscan.shape[1]
    w_end   = make_end_taper(n_t, int(round(taper_end_ns / dt_ns)))
    w_decay = make_decay_taper(time_ns, taper_decay_ns)
    tapered = bscan * (w_end * w_decay)[np.newaxis, :]
    t0_samp = int(round(t0_ns / dt_ns))
    shifted = np.roll(tapered, -t0_samp, axis=1)
    shifted[:, -t0_samp:] = 0.0
    return tapered, shifted

In [ ]:
# ── Gazdag phase-shift migration on all 7 datasets ────────────────────────────────
migrated_gz = {}

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Gazdag ...', flush=True)
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) — time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz[label] = img


In [ ]:
# ── Plot: full extent ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)   # 7 items in 2×4 grid — hide last slot
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_gz.items())):
    vmax = np.max(np.abs(img)) * 0.8
    ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_title(f'Gazdag — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')

plt.suptitle(
    f'Gazdag Phase-Shift Migration — All 7 Datasets  |  f_c={f_c_GHz} GHz  |  z_top={z_top:.3f} m',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
# plt.savefig(STUDY_ROOT / 'gazdag_all_datasets.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot: zoomed on scatterer region ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True, sharey=True)
axes.ravel()[-1].set_visible(False)
for i, (ax, (label, img)) in enumerate(zip(axes.ravel(), migrated_gz.items())):
    vmax = np.max(np.abs(img)) * 0.8
    ax.imshow(img, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0], z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i], z_top, 'g^', ms=10, zorder=5, label='x_s1 current')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'{label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Gazdag Phase-Shift Migration (zoomed)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
# plt.savefig(STUDY_ROOT / 'gazdag_all_datasets_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Gazdag timelapse differences (migrated − migrated_baseline) ──────────────────────
migrated_gz_diff = {label: migrated_gz[label] - migrated_gz['Baseline'] for label in labels}

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_gz_diff.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_title(f'TimeLapse diff — {label}', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Gazdag Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
# plt.savefig(STUDY_ROOT / 'gazdag_timelapse_diff.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Zoomed ────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
for i, (ax, (label, diff)) in enumerate(zip(axes.ravel(), migrated_gz_diff.items())):
    vmax = np.max(np.abs(diff)) * 0.8
    ax.imshow(diff, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
              extent=extent_mig, origin='upper')
    ax.plot(x_scatterer_1[0],     z_top, 'g*', ms=10, zorder=5, label='x_s1 baseline')
    ax.plot(x_scatterer_1[i + 1], z_top, 'g^', ms=10, zorder=5, label='x_s1 timelapsed')
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.75, 0.55)
    ax.set_title(f'TimeLapse diff — {label} (zoomed)', fontsize=11)
    ax.set_xlabel('x [m]'); ax.set_ylabel('Depth z [m]')
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(
    f'Gazdag Migration — TimeLapse Differences (zoomed)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
# plt.savefig(STUDY_ROOT / 'gazdag_timelapse_diff_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from scipy.signal import hilbert as sp_hilbert

# Dataset to analyse  (<-- tune)
LABEL   = '¼λ'
SC1_IDX = 4      # index into x_scatterer_1 for ground-truth only

# ── Ground truth (simulation parameters, NOT fed to the estimator) ─────────────
dl = 0.002   # gprMax FDTD cell size [m]; scatterer positions snap to this grid
x_s1_base_g = np.round(x_scatterer_1[0]       / dl) * dl
x_s1_mon_g  = np.round(x_scatterer_1[SC1_IDX] / dl) * dl
true_dz = 0.0
true_dx = float(x_s1_mon_g - x_s1_base_g)

dz_mig = z_img[1]    - z_img[0]
dx_mig = x_traces[1] - x_traces[0]
kz_c   = 2 * np.pi / wavelength

# ── Locate moving scatterer from the data (no knowledge of x_scatterer_1) ─────
# The difference envelope is large only where a scatterer moved; the fixed
# scatterer at x_centre + 3λ cancels and does not contribute a peak.
env_base = np.abs(sp_hilbert(migrated_gz["Baseline"], axis=0))
env_mon  = np.abs(sp_hilbert(migrated_gz[LABEL],      axis=0))
diff_env = np.abs(env_mon - env_base)

# Rough location from the difference peak (may be offset ~σ from apex for small shifts)
iz_d, ix_d = np.unravel_index(np.argmax(diff_env), diff_env.shape)
x_rough = x_traces[ix_d]

# Refine: find the baseline envelope apex within ±3λ of the rough location
hw_search = 3 * wavelength
ix_lo_s   = np.searchsorted(x_traces, x_rough - hw_search)
ix_hi_s   = np.searchsorted(x_traces, x_rough + hw_search)
_, ix_loc = np.unravel_index(np.argmax(env_base[:, ix_lo_s:ix_hi_s]),
                              env_base[:, ix_lo_s:ix_hi_s].shape)
x_apex = x_traces[ix_lo_s + ix_loc]

# ── Crop around the data-found apex ───────────────────────────────────────────
x_crop_hw = 2.5 * wavelength # 3.5 also worked in this case
ix_lo = np.searchsorted(x_traces, x_apex - x_crop_hw)
ix_hi = np.searchsorted(x_traces, x_apex + x_crop_hw)
x_crop = x_traces[ix_lo:ix_hi]

base_crop = migrated_gz["Baseline"][:, ix_lo:ix_hi]
mon_crop  = migrated_gz[LABEL][:,     ix_lo:ix_hi]

# ── Estimate ────────────────────────────────────────────────────────────────────────────────
dz_est, dx_est, XS, kz_ax, kx_ax = estimate_shift_2d(
    base_crop, mon_crop, dz_mig, dx_mig, kz_c
)

XS_s     = np.fft.fftshift(XS)
kz_s     = np.fft.fftshift(kz_ax)
kx_s     = np.fft.fftshift(kx_ax)
energy   = np.abs(XS_s)
phi_show = np.where(energy > 0.01 * energy.max(),
                    np.degrees(np.angle(XS_s)), np.nan)

# ── Plot ──────────────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle(
    f'2D cross-spectrum — Horizontal shift  |  dataset: {LABEL}'
    f'Scatterer located from data at x = {x_apex*100:.2f} cm  '
    f'(ground truth: {x_s1_base_g*100:.2f} cm)',
    fontsize=11
)

# Col 0: difference image over cropped region
ax = axes[0]
diff = mon_crop - base_crop
vmax = np.max(np.abs(diff))
ax.pcolormesh(x_crop * 100, z_img * 100, diff,
              cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
ax.axvline(x_apex          * 100, color='k', ls='-',  lw=1.5, label='apex (data)')
ax.axvline(x_s1_base_g     * 100, color='g', ls='--', lw=1,   label='x_s1 base (GT)')
ax.axvline(x_s1_mon_g      * 100, color='r', ls='--', lw=1,   label=f'x_s1 {LABEL} (GT)')
ax.set_title(f'Difference (mon − base) dx_true = {true_dx*1e3:.2f} mm  (GT, FDTD-snapped)', fontsize=9)
ax.set_xlabel('x [cm]');  ax.set_ylabel('z [cm]')
ax.invert_yaxis();  ax.legend(fontsize=8)

klim = 1.4 * kz_c

# Col 1: cross-spectrum phase
ax2 = axes[1]
im2 = ax2.pcolormesh(kx_s, kz_s, phi_show,
                     cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
for sgn in [-1, 1]:
    ax2.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
    ax2.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
ax2.set_title('Cross-spectrum phase [deg] dashed = fit band', fontsize=9)
ax2.set_xlabel('kx [rad/m]');  ax2.set_ylabel('kz [rad/m]')
ax2.set_xlim(kx_s.min() / 2, kx_s.max() / 2)
ax2.set_ylim(kz_s.min() / 2, kz_s.max() / 2)
plt.colorbar(im2, ax=ax2, fraction=0.046)

# Col 2: energy |XS|
ax4 = axes[2]
im4 = ax4.pcolormesh(kx_s, kz_s, energy, cmap='inferno', shading='auto')
for sgn in [-1, 1]:
    ax4.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.6)
    ax4.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.6)
ax4.set_title('Cross-spectrum energy |XS| dashed = fit band', fontsize=9)
ax4.set_xlabel('kx [rad/m]');  ax4.set_ylabel('kz [rad/m]')
ax4.set_xlim(kx_s.min() / 2, kx_s.max() / 2)
ax4.set_ylim(kz_s.min() / 2, kz_s.max() / 2)
plt.colorbar(im4, ax=ax4, fraction=0.046, label='|XS|')

# Col 3: results
ax3 = axes[3]
ax3.axis('off')
txt = (
    f'True:  dz = {true_dz * 1e3:+7.3f} mm\n'
    f'       dx = {true_dx * 1e3:+7.3f} mm\n\n'
    f'Est:   dz = {dz_est  * 1e3:+7.3f} mm\n'
    f'       dx = {dx_est  * 1e3:+7.3f} mm\n\n'
    f'Err:   dz = {(dz_est - true_dz) * 1e3:+.4f} mm\n'
    f'       dx = {(dx_est - true_dx) * 1e3:+.4f} mm\n\n'
    f'Apex found at  {x_apex*100:.3f} cm\n'
    f'GT scatterer:  {x_s1_base_g*100:.3f} cm'
)
ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
         fontsize=10, family='monospace', verticalalignment='top')
ax3.set_title('Results', fontsize=9)

plt.tight_layout()
plt.show()

# Material Change Test

In [ ]:
from scipy.signal import fftconvolve
from scipy.signal import hilbert as sp_hilbert
from scipy.signal.windows import tukey

# ── Physics ────────────────────────────────────────────────────────────────────
c_light   = 0.299792458          # m / ns
eps_ice   = 3.15
n_ice     = np.sqrt(eps_ice)
n_air     = 1.0                  # baseline fill
n_water   = 9.0                  # sqrt(81), monitor fill

v_ice     = c_light / n_ice      # ≈ 0.1689 m/ns
f_c       = 1.5                  # GHz
lam       = v_ice / f_c          # ≈ 0.1126 m
v_mig     = v_ice / 2
d         = lam / 20             # fracture thickness ≈ 5.6 mm

dz_mig = lam / 20
dx_mig = lam / 4
kz_c   = 2 * np.pi / lam

Nz, Nx = 256, 64
z_mc   = np.arange(Nz) * dz_mig
x_mc   = np.arange(Nx) * dx_mig

# ── Fresnel reflection coefficients ───────────────────────────────────────────
def refl(n1, n2):
    return (n1 - n2) / (n1 + n2)

R_top_air   = refl(n_ice, n_air)    # ice → air   ≈ +0.279
R_bot_air   = refl(n_air, n_ice)    # air → ice   ≈ -0.279
R_top_water = refl(n_ice, n_water)  # ice → water ≈ -0.671  (polarity reversal)
R_bot_water = refl(n_water, n_ice)  # water → ice ≈ +0.671

dz_del_air   = d * n_air   / n_ice
dz_del_water = d * n_water / n_ice

iz_del_air   = max(1, int(round(dz_del_air   / dz_mig)))
iz_del_water = max(1, int(round(dz_del_water / dz_mig)))

# ── Ricker wavelet (migration-depth domain) ────────────────────────────────────
Nw   = 2 * int(round(3.5 * lam / dz_mig)) + 1
z_wl = (np.arange(Nw) - Nw // 2) * dz_mig
u_wl = (np.pi * f_c * z_wl / v_mig) ** 2
w_rk = (1 - 2 * u_wl) * np.exp(-u_wl)

# ── Reflectivity series & 1D traces ───────────────────────────────────────────
iz0 = Nz // 3

r_base = np.zeros(Nz)
r_base[iz0]              += R_top_air
r_base[iz0 + iz_del_air] += R_bot_air

r_mon = np.zeros(Nz)
r_mon[iz0]                += R_top_water
r_mon[iz0 + iz_del_water] += R_bot_water

trace_base = fftconvolve(r_base, w_rk, mode='same')
trace_mon  = fftconvolve(r_mon,  w_rk, mode='same')

# ── 2D migrated B-scans ────────────────────────────────────────────────────────
# Outer product of 1D depth trace with a lateral Gaussian (sigma_x = lam/2).
# This models the focused PSF of a Gazdag-migrated horizontal reflector.
# A Ricker wavelet in depth  x  Gaussian in x  =  the migrated image.
sigma_x = lam / 2
gauss_x = np.exp(-0.5 * ((x_mc - x_mc.mean()) / sigma_x) ** 2)

base_bscan = np.outer(trace_base, gauss_x)   # shape (Nz, Nx)
mon_bscan  = np.outer(trace_mon,  gauss_x)

# ── ROI: include full Ricker wavelet on both sides of fracture ─────────────────
n_margin = Nw // 2 + 5
iz_lo    = max(0,  iz0 - n_margin)
iz_hi    = min(Nz, iz0 + iz_del_water + n_margin + 1)
z_roi    = z_mc[iz_lo:iz_hi]

base_roi = base_bscan[iz_lo:iz_hi, :]
mon_roi  = mon_bscan[iz_lo:iz_hi, :]
diff_roi = mon_roi - base_roi

# ── Analytic signal along z ────────────────────────────────────────────────────
# Cross-spectra of real images are Hermitian: phi(-k)=-phi(k),
# forcing phi_0=0 in the WLS fit regardless of material change.
# Hilbert along z removes negative-kz components, enabling phi_0 != 0.
a_base = sp_hilbert(base_roi, axis=0)
a_mon  = sp_hilbert(mon_roi,  axis=0)

# ── Shift estimate ─────────────────────────────────────────────────────────────
dz_est, dx_est, XS, kz_ax, kx_ax = estimate_shift_2d(
    a_base, a_mon, dz_mig, dx_mig, kz_c
)

# Re-run WLS for phase intercept c[2]
KZ_f, KX_f = np.meshgrid(kz_ax, kx_ax, indexing='ij')
w_f    = np.abs(XS);  phi_f = np.angle(XS)
band_f = (np.abs(KZ_f) < 1.4 * kz_c) & (np.abs(KX_f) < 1.4 * kz_c)
mask_f = (w_f > 0.1 * w_f.max()) & band_f & ((np.abs(KZ_f) + np.abs(KX_f)) > 0)
A_f    = np.column_stack([KZ_f[mask_f], KX_f[mask_f], np.ones(mask_f.sum())])
W_f    = w_f[mask_f]
c_all  = np.linalg.lstsq(A_f * W_f[:, None], phi_f[mask_f] * W_f, rcond=None)[0]
phi_0  = c_all[2]

# ── Figure: 2x3 layout ────────────────────────────────────────────────────────
XS_s    = np.fft.fftshift(XS)
kz_s    = np.fft.fftshift(kz_ax)
kx_s    = np.fft.fftshift(kx_ax)
energy  = np.abs(XS_s)
phi_vis = np.where(energy > 0.01 * energy.max(),
                   np.degrees(np.angle(XS_s)), np.nan)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(
    f'Material Change Test — horizontal fracture  d = λ/20 = {d*1e3:.1f} mm'
    f'Baseline: Air (ε = {n_air**2:.0f})   →   Monitor: Water (ε = {n_water**2:.0f})'
    '   |   no physical movement',
    fontsize=11
)

x_cm = x_mc * 100;  z_cm = z_roi * 100
z_frac_top = z_mc[iz0] * 100
z_frac_bot = z_mc[iz0 + iz_del_water] * 100

def _bscan_ax(ax, img, title, vmax=None):
    vmax = vmax or np.max(np.abs(img))
    ax.pcolormesh(x_cm, z_cm, img,
                  cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
    ax.axhline(z_frac_top, color='gold',  lw=1.2, ls='--', label='fracture top')
    ax.axhline(z_frac_bot, color='olive', lw=1.2, ls=':',  label='fracture bot (H₂O)')
    ax.set_xlabel('x [cm]');  ax.set_ylabel('Depth [cm]')
    ax.invert_yaxis()
    ax.set_title(title, fontsize=9)

# (0,0) Baseline migrated B-scan
vmax_b = np.max(np.abs(base_roi))
_bscan_ax(axes[0, 0], base_roi,
          f'Baseline migrated B-scan Air fill  R_top = {R_top_air:+.3f}', vmax_b)
axes[0, 0].legend(fontsize=7, loc='lower right')

# (0,1) Monitor migrated B-scan (same colour scale for direct comparison)
_bscan_ax(axes[0, 1], mon_roi,
          f'Monitor migrated B-scan Water fill  R_top = {R_top_water:+.3f}', vmax_b)

# (0,2) Difference: monitor – baseline
vmax_d = np.max(np.abs(diff_roi))
_bscan_ax(axes[0, 2], diff_roi,
          'Difference  (Monitor − Baseline) Reveals polarity & depth change', vmax_d)

# (1,0) Central trace overlay
ax = axes[1, 0]
norm_b = np.max(np.abs(trace_base));  norm_m = np.max(np.abs(trace_mon))
ax.plot(trace_base[iz_lo:iz_hi] / norm_b, z_cm,
        color='C0', lw=1.8, label=f'Baseline Air   (R_top={R_top_air:+.3f})')
ax.plot(trace_mon[iz_lo:iz_hi]  / norm_m, z_cm,
        color='C1', lw=1.8, ls='--',
        label=f'Monitor Water  (R_top={R_top_water:+.3f})')
ax.axhline(z_frac_top, color='gold',  lw=1.2, ls='--')
ax.axhline(z_frac_bot, color='olive', lw=1.2, ls=':')
ax.set_xlabel('Normalised amplitude');  ax.set_ylabel('Depth [cm]')
ax.set_title('Central trace (normalised) Note polarity reversal & pulse-shape change', fontsize=9)
ax.invert_yaxis();  ax.legend(fontsize=8);  ax.grid(alpha=0.2)

# (1,1) Cross-spectrum phase
ax2 = axes[1, 1]
im  = ax2.pcolormesh(kx_s, kz_s, phi_vis,
                     cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
klim = 1.4 * kz_c
for sgn in [-1, 1]:
    ax2.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.6)
    ax2.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.6)
ax2.set_title(
    'Analytic cross-spectrum phase [deg]'
    'Approx. flat ≈ constant polarity offset'
    'Small kz slope → apparent shift from Δv', fontsize=9
)
ax2.set_xlabel('kx [rad/m]');  ax2.set_ylabel('kz [rad/m]')
ax2.set_xlim(-2 * kz_c, 2 * kz_c);  ax2.set_ylim(-2 * kz_c, 2 * kz_c)
plt.colorbar(im, ax=ax2, fraction=0.046)

# (1,2) Results
ax3 = axes[1, 2]
ax3.axis('off')
txt = (
    'Fracture model:'
    f'  d          = λ/20 = {d*1e3:.2f} mm\n'
    f'  Baseline: Air    R_top = {R_top_air:+.3f}\n'
    f'  Monitor:  Water  R_top = {R_top_water:+.3f}\n'
    'Depth delay in migrated image:\n'
    f'  Air:   Δz_bot = {dz_del_air*1e3:.1f} mm  (< 1 sample)\n'
    f'  Water: Δz_bot = {dz_del_water*1e3:.1f} mm  (≈ {iz_del_water} samples)\n'
    'Estimated (true displacement = 0):\n'
    f'  Δz = {dz_est*1e3:+.1f} mm   [apparent shift from Δv]\n'
    f'  Δx = {dx_est*1e3:+.1f} mm   [correct: 0]\n'
    'Material change signature:'
    f'  c = {np.degrees(phi_0):+.1f}°   [polarity reversal + thin-layer phase]\n'
)
ax3.text(0.04, 0.97, txt, transform=ax3.transAxes,
         fontsize=10, family='monospace', verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.6))
ax3.set_title('Results', fontsize=9)

plt.tight_layout()
plt.show()

print(f'dz={dz_est*1e3:+.1f} mm  dx={dx_est*1e3:+.1f} mm  phi_0={np.degrees(phi_0):+.1f} deg')


# Moving Water Front Test

In [ ]:
from scipy.signal import fftconvolve
from scipy.signal import hilbert as sp_hilbert
from matplotlib.patches import Rectangle

# ── Physics ──────────────────────────────────────────────────────────────────────
c_light_wf  = 0.299792458
eps_ice_wf  = 3.15;  n_ice_wf = np.sqrt(eps_ice_wf)
n_air_wf    = 1.0;   n_water_wf = 9.0
v_ice_wf    = c_light_wf / n_ice_wf   # ≈ 0.1689 m/ns
f_c_wf      = 1.5                      # GHz
lam_wf      = v_ice_wf / f_c_wf       # ≈ 0.1126 m
kz_c_wf     = 2 * np.pi / lam_wf

# ── Grid ─────────────────────────────────────────────────────────────────────────
Nx_wf     = 128 * 2;  dx_wf = lam_wf / 16
x_wf      = np.arange(Nx_wf) * dx_wf  # [m]

# dt chosen so one time-sample == one depth-sample after migration
dz_wf     = lam_wf / 40
v_mig_wf  = v_ice_wf / 2
dt_wf     = dz_wf / v_mig_wf               # ≈ 0.0333 ns
Nt_wf     = 1024 * 2
t_wf      = np.arange(Nt_wf) * dt_wf  # [ns]

Nz_wf     = 512 * 2
z_wf      = np.arange(Nz_wf) * dz_wf  # [m]

# ── Fracture geometry ─────────────────────────────────────────────────────────────
iz0_wf       = Nz_wf // 3              # fracture top index in depth axis
z0_wf        = z_wf[iz0_wf]           # ≈ 0.48 m
d_frac_wf    = lam_wf / 20            # fracture thickness ≈ 5.6 mm

x_front_base = x_wf[Nx_wf // 2]      # water front in baseline (domain midpoint)
delta_x_true = lam_wf / 8             # subwavelength shift
x_front_mon  = x_front_base + delta_x_true

# ── Reflection coefficients ───────────────────────────────────────────────────────
def _refl(n1, n2): return (n1 - n2) / (n1 + n2)
R_top_air_wf   = _refl(n_ice_wf, n_air_wf)
R_bot_air_wf   = _refl(n_air_wf,   n_ice_wf)
R_top_water_wf = _refl(n_ice_wf, n_water_wf)
R_bot_water_wf = _refl(n_water_wf, n_ice_wf)
dR_wf          = R_top_water_wf - R_top_air_wf  # edge diffraction amplitude

# Integer sample delays through fracture
# Since dt_wf = dz_wf/v_mig_wf and the thin-layer delay = d*n_fill/n_ice / dz_wf,
# time delay in samples == depth delay in samples (1:1 mapping).
iz_del_air_wf   = max(1, int(round(d_frac_wf * n_air_wf   / n_ice_wf / dz_wf)))
iz_del_water_wf = max(1, int(round(d_frac_wf * n_water_wf / n_ice_wf / dz_wf)))

# ── Ricker wavelet (time domain) ──────────────────────────────────────────────────
Nw_t_wf   = 2 * int(round(3.5 / f_c_wf / dt_wf)) + 1
t_wl_wf   = (np.arange(Nw_t_wf) - Nw_t_wf // 2) * dt_wf
w_rk_t_wf = (1 - 2*(np.pi*f_c_wf*t_wl_wf)**2) * np.exp(-(np.pi*f_c_wf*t_wl_wf)**2)

# ── B-scan synthesis: flat reflector + edge diffraction from fill boundary ────────
def _make_bscan_wf(x_front):
    """
    Per-trace reflectivity determined by fill at that x (water left, air right).
    Edge diffraction from the fill step at x_front is added via point_bscan,
    amplitude-normalised so the hyperbola apex == dR_wf (the fill impedance step).
    """
    bscan = np.zeros((Nt_wf, Nx_wf))
    it0 = int(round(2.0 * z0_wf / v_ice_wf / dt_wf))  # == iz0_wf
    for ix in range(Nx_wf):
        if x_wf[ix] < x_front:
            Rt, Rb, it_d = R_top_water_wf, R_bot_water_wf, iz_del_water_wf
        else:
            Rt, Rb, it_d = R_top_air_wf,   R_bot_air_wf,   iz_del_air_wf
        r = np.zeros(Nt_wf)
        r[it0]        += Rt
        r[it0 + it_d] += Rb
        bscan[:, ix] = fftconvolve(r, w_rk_t_wf, mode='same')
    # Edge diffraction: point_bscan gives 1/sqrt(r) spreading;
    # multiply by sqrt(z0) so amplitude at apex (r=z0) equals dR_wf.
    edge = dR_wf * np.sqrt(z0_wf) * point_bscan(
        x_front, z0_wf, x_wf, t_wf, v_ice_wf, f_c_wf
    )
    return bscan + edge

bscan_base_wf = _make_bscan_wf(x_front_base)
bscan_mon_wf  = _make_bscan_wf(x_front_mon)

# ── Gazdag migration ──────────────────────────────────────────────────────────────
print('Migrating baseline ...')
mig_base_wf = gazdag_migration(bscan_base_wf, x_wf, t_wf, z_wf, v_ice_wf)
print('Migrating monitor  ...')
mig_mon_wf  = gazdag_migration(bscan_mon_wf,  x_wf, t_wf, z_wf, v_ice_wf)
diff_mig_wf = mig_mon_wf - mig_base_wf

# ── Cross-spectrum ROI ────────────────────────────────────────────────────────────
# z: wavelet half-width on each side of the fracture
Nw_z_wf  = 2 * int(round(3.5 * lam_wf / dz_wf)) + 1
n_mz_wf  = Nw_z_wf // 2 + 5
iz_lo_wf = max(0,     iz0_wf - n_mz_wf)
iz_hi_wf = min(Nz_wf, iz0_wf + iz_del_water_wf + n_mz_wf + 1)

# x: ±4λ centred on baseline water front (edge diffraction dominates there)
n_mx_wf  = int(round(4.0 * lam_wf / dx_wf)) + 1
ix_c_wf  = int(np.argmin(np.abs(x_wf - x_front_base)))
ix_lo_wf = max(0,     ix_c_wf - n_mx_wf)
ix_hi_wf = min(Nx_wf, ix_c_wf + n_mx_wf + 1)

roi_base_wf = mig_base_wf[iz_lo_wf:iz_hi_wf, ix_lo_wf:ix_hi_wf]
roi_mon_wf  = mig_mon_wf [iz_lo_wf:iz_hi_wf, ix_lo_wf:ix_hi_wf]

# Analytic signal along z (breaks Hermitian symmetry → enables phi_0 estimation)
a_base_wf = sp_hilbert(roi_base_wf, axis=0)
a_mon_wf  = sp_hilbert(roi_mon_wf,  axis=0)

dz_est_wf, dx_est_wf, XS_wf, kz_ax_wf, kx_ax_wf = estimate_shift_2d(
    a_base_wf, a_mon_wf, dz_wf, dx_wf, kz_c_wf
)

# Re-run WLS for phase intercept phi_0
KZ_wf, KX_wf_g = np.meshgrid(kz_ax_wf, kx_ax_wf, indexing='ij')
w_wf   = np.abs(XS_wf);  phi_wf = np.angle(XS_wf)
band_wf = (np.abs(KZ_wf) < 1.4*kz_c_wf) & (np.abs(KX_wf_g) < 1.4*kz_c_wf)
mask_wf = (w_wf > 0.1*w_wf.max()) & band_wf & ((np.abs(KZ_wf)+np.abs(KX_wf_g)) > 0)
A_wf    = np.column_stack([KZ_wf[mask_wf], KX_wf_g[mask_wf], np.ones(mask_wf.sum())])
W_wf    = w_wf[mask_wf]
c_wf    = np.linalg.lstsq(A_wf*W_wf[:,None], phi_wf[mask_wf]*W_wf, rcond=None)[0]
phi_0_wf = c_wf[2]

In [ ]:

# ── Figure: 3 × 2 layout ─────────────────────────────────────────────────────────
t_ns = t_wf.copy()                 # already in ns
x_cm = x_wf * 100
z_cm = z_wf * 100

z0_cm    = z0_wf * 100
xfb_cm   = x_front_base * 100
xfm_cm   = x_front_mon  * 100
roi_zlo  = z_wf[iz_lo_wf]         * 100
roi_zhi  = z_wf[iz_hi_wf - 1]     * 100
roi_xlo  = x_wf[ix_lo_wf]         * 100
roi_xhi  = x_wf[ix_hi_wf - 1]     * 100
t0_ns_v  = 2.0 * z0_wf / v_ice_wf

fig, axes = plt.subplots(3, 2, figsize=(16, 18), dpi=120)
fig.suptitle(
    f'Moving Water Front Test — fracture full lateral extent,  d = \u03bb/20 = {d_frac_wf*1e3:.1f} mm\n'
    f'Baseline: water front at x = {xfb_cm:.1f} cm   \u2192   '
    f'Monitor: front shifted by \u03b4x = \u03bb/8 = {delta_x_true*1e3:.1f} mm',
    fontsize=11
)

# Row 0: raw time-domain B-scans
vmax_raw = max(np.max(np.abs(bscan_base_wf)), np.max(np.abs(bscan_mon_wf)))
for col, (bscan, title, xf_cm) in enumerate([
    (bscan_base_wf, 'Raw baseline B-scan (time domain)\nHyperbola centred at water front', xfb_cm),
    (bscan_mon_wf,  'Raw monitor B-scan (time domain)\nHyperbola shifted by \u03b4x = \u03bb/8', xfm_cm),
]):
    ax = axes[0, col]
    ax.pcolormesh(x_cm, t_ns, bscan, cmap='RdBu_r',
                  vmin=-vmax_raw, vmax=vmax_raw, shading='auto')
    ax.axhline(t0_ns_v, color='gold', lw=1.2, ls='--',
               label=f'TWT(z\u2080) = {t0_ns_v:.2f} ns')
    ax.axvline(xf_cm, color='cyan', lw=1.2, ls=':',
               label=f'x_front = {xf_cm:.1f} cm')
    ax.set_xlabel('x [cm]');  ax.set_ylabel('Two-way time [ns]')
    ax.set_ylim(8, 17)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=9);  ax.legend(fontsize=7)

# Row 1: migrated images (common colour scale)
vmax_mig = max(np.max(np.abs(mig_base_wf)), np.max(np.abs(mig_mon_wf)))
for col, (img, title, xf_cm) in enumerate([
    (mig_base_wf, 'Migrated baseline\nHyperbola collapsed to focused edge', xfb_cm),
    (mig_mon_wf,  'Migrated monitor\nFocused edge shifted to new front',    xfm_cm),
]):
    ax = axes[1, col]
    ax.pcolormesh(x_cm, z_cm, img, cmap='RdBu_r',
                  vmin=-vmax_mig, vmax=vmax_mig, shading='auto')
    ax.axhline(z0_cm, color='gold', lw=1.2, ls='--', label='z\u2080 (fracture)')
    ax.axvline(xf_cm, color='cyan', lw=1.2, ls=':',
               label=f'x_front = {xf_cm:.1f} cm')
    ax.set_xlabel('x [cm]');  ax.set_ylabel('Depth [cm]')
    ax.set_ylim(60, 150)
    ax.invert_yaxis();  ax.set_title(title, fontsize=9);  ax.legend(fontsize=7)

# Row 2, left: migrated difference with ROI box
ax = axes[2, 0]
vmax_d = np.max(np.abs(diff_mig_wf)) or 1.0
ax.pcolormesh(x_cm, z_cm, diff_mig_wf, cmap='RdBu_r',
              vmin=-vmax_d, vmax=vmax_d, shading='auto')
ax.axhline(z0_cm,  color='gold',  lw=1.2, ls='--')
ax.axvline(xfb_cm, color='cyan',  lw=1.2, ls=':', label=f'x_front base = {xfb_cm:.1f} cm')
ax.axvline(xfm_cm, color='lime',  lw=1.2, ls=':', label=f'x_front mon  = {xfm_cm:.1f} cm')
roi_rect = Rectangle(
    (roi_xlo, roi_zlo), roi_xhi - roi_xlo, roi_zhi - roi_zlo,
    edgecolor='yellow', facecolor='none', lw=1.8
)
ax.add_patch(roi_rect)
ax.set_xlabel('x [cm]');  ax.set_ylabel('Depth [cm]')
ax.set_ylim(40, 160)
ax.invert_yaxis()
ax.set_title(
    'Migrated difference  (Monitor \u2212 Baseline)\n'
    '[yellow box = ROI for phase-plane fit]', fontsize=9
)
ax.legend(fontsize=7)

# Row 2, right: results text
ax3 = axes[2, 1];  ax3.axis('off')
err_dx = (dx_est_wf - delta_x_true) * 1e3
txt = (
    'Model:\n'
    f'  Fill: water (left)  |  air (right)\n'
    f'  R_top water = {R_top_water_wf:+.3f}   R_top air = {R_top_air_wf:+.3f}\n'
    f'  \u0394R (edge) = {dR_wf:+.3f}\n'
    f'  d = \u03bb/20 = {d_frac_wf*1e3:.1f} mm\n\n'
    'True displacement:\n'
    f'  \u03b4x = \u03bb/8 = {delta_x_true*1e3:.2f} mm\n'
    f'  \u03b4z = 0 mm\n\n'
    'Estimated (phase-plane fit on ROI):\n'
    f'  \u0394z = {dz_est_wf*1e3:+.1f} mm   [true: 0]\n'
    f'  \u0394x = {dx_est_wf*1e3:+.1f} mm   [true: {delta_x_true*1e3:.1f} mm]\n'
    f'  Error = {err_dx:+.1f} mm\n\n'
    f'  \u03c6\u2080 = {np.degrees(phi_0_wf):+.1f}\u00b0\n'
    f'  [partial polarity change in \u03b4x = {delta_x_true/lam_wf:.3f}\u03bb strip]'
)
ax3.text(0.04, 0.97, txt, transform=ax3.transAxes,
         fontsize=10, family='monospace', verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.6))
ax3.set_title('Results', fontsize=9)

plt.tight_layout()
plt.show()

print(f'True \u03b4x = {delta_x_true*1e3:.2f} mm  |  '
      f'Estimated \u0394x = {dx_est_wf*1e3:.2f} mm  |  '
      f'Error = {err_dx:+.2f} mm')


# Two-Step Polarity Retrieval

In [ ]:
from scipy.signal import hilbert as sp_hilbert

# Reuses from Material Change Test cell:
#   lam, dz_mig, dx_mig, kz_c, x_mc, sigma_x, iz_lo, iz_hi
#   trace_base (air fill), trace_mon (water fill)
#   R_top_air, R_top_water, estimate_shift_2d

# ── Scenario: fill change (air->water) + lateral shift delta_x = lam/4 ──────────
delta_x_p = lam / 16                 # shift = 1 spatial sample (dx_mig)
gauss_ctr = x_mc.mean()

gauss_b_p = np.exp(-0.5 * ((x_mc - gauss_ctr            ) / sigma_x)**2)
gauss_m_p = np.exp(-0.5 * ((x_mc - gauss_ctr - delta_x_p) / sigma_x)**2)

base_p = np.outer(trace_base, gauss_b_p)   # air fill, centred
mon_p  = np.outer(trace_mon,  gauss_m_p)   # water fill, shifted right

roi_b_p = base_p[iz_lo:iz_hi, :]
roi_m_p = mon_p [iz_lo:iz_hi, :]

a_b_p = sp_hilbert(roi_b_p, axis=0)
a_m_p = sp_hilbert(roi_m_p, axis=0)

# ── Step 1: joint phase-plane fit  ->  Delta_z, Delta_x, phi_0 (naive) ───────────
dz_p, dx_p, XS_p, kz_p, kx_p = estimate_shift_2d(a_b_p, a_m_p, dz_mig, dx_mig, kz_c)

KZ_p, KX_p = np.meshgrid(kz_p, kx_p, indexing='ij')
w_p    = np.abs(XS_p);  phi_p = np.angle(XS_p)
band_p = (np.abs(KZ_p) < 1.4*kz_c) & (np.abs(KX_p) < 1.4*kz_c)
mask_p = (w_p > 0.1*w_p.max()) & band_p & ((np.abs(KZ_p)+np.abs(KX_p)) > 0)
A_p    = np.column_stack([KZ_p[mask_p], KX_p[mask_p], np.ones(mask_p.sum())])
W_p    = w_p[mask_p]
c_p    = np.linalg.lstsq(A_p*W_p[:,None], phi_p[mask_p]*W_p, rcond=None)[0]
phi_0_s1 = c_p[2]

# ── Step 2: divide out the estimated shift ramp -> isolate phi_0 ──────────────────
# Multiply XS by exp(-i*(kz*Dz + kx*Dx)) to remove the displacement phase.
# What remains is the constant polarity offset phi_0.
XS_corr  = XS_p * np.exp(-1j * (KZ_p * dz_p + KX_p * dx_p))
phi_corr = np.angle(XS_corr)
phi_0_s2 = np.average(phi_corr[mask_p], weights=W_p)

# ── Figure: 1x3 ───────────────────────────────────────────────────────────────────
kz_s = np.fft.fftshift(kz_p)
kx_s = np.fft.fftshift(kx_p)
energy_s = np.fft.fftshift(np.abs(XS_p))
thresh   = 0.01 * energy_s.max()

vis_raw  = np.where(energy_s > thresh,
                    np.degrees(np.angle(np.fft.fftshift(XS_p))),    np.nan)
vis_corr = np.where(energy_s > thresh,
                    np.degrees(np.angle(np.fft.fftshift(XS_corr))), np.nan)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
fig.suptitle(
    'Two-step polarity retrieval  \u2014  fill change (air\u2192water, \u03c6\u2080 = \u2212180\u00b0) '
    '+ lateral shift \u03b4x = \u03bb/4\n'
    'Step 1: joint phase-plane fit extracts shift  |  '
    'Step 2: remove shift ramp \u2192 flat phase \u2248 \u2212180\u00b0',
    fontsize=10
)

kw   = dict(cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
klim = 1.4 * kz_c

for ax, vis, ttl in [
    (axes[0], vis_raw,
     'Step 1: raw cross-spectrum phase\n'
     'Tilted pattern: kx slope from shift,  offset from fill change'),
    (axes[1], vis_corr,
     f'Step 2: shift ramp removed  (\u0394x = {dx_p*1e3:.1f} mm)\n'
     'Flat pattern \u2248 \u2212180\u00b0  \u2192  polarity reversal confirmed'),
]:
    im = ax.pcolormesh(kx_s, kz_s, vis, **kw)
    for sgn in [-1, 1]:
        ax.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlim(-2*kz_c, 2*kz_c);  ax.set_ylim(-2*kz_c, 2*kz_c)
    ax.set_xlabel('kx [rad/m]');    ax.set_ylabel('kz [rad/m]')
    ax.set_title(ttl, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, label='Phase [deg]')

ax3 = axes[2];  ax3.axis('off')
txt = (
    'Scenario:\n'
    f'  Baseline: air fill    R_top = {R_top_air:+.3f}\n'
    f'  Monitor:  water fill  R_top = {R_top_water:+.3f}\n'
    f'            + shift \u03b4x = \u03bb/4 = {delta_x_p*1e3:.1f} mm\n\n'
    'Step 1 \u2014 joint phase-plane fit:\n'
    f'  \u0394z = {dz_p*1e3:+.1f} mm   [true: 0]\n'
    f'  \u0394x = {dx_p*1e3:+.1f} mm   [true: {delta_x_p*1e3:.1f}]\n'
    f'  \u03c6\u2080 = {np.degrees(phi_0_s1):+.1f}\u00b0   (simultaneous fit)\n\n'
    'Step 2 \u2014 remove shift ramp from XS:\n'
    f'  \u03c6\u2080 = {np.degrees(phi_0_s2):+.1f}\u00b0\n\n'
    'Discriminator:\n'
    '  |\u03c6\u2080| \u2248 180\u00b0  \u2192  fill change\n'
    '  |\u03c6\u2080| \u2248   0\u00b0  \u2192  pure displacement\n'
    '  (cf. water-front test: \u03c6\u2080 \u2248 +0.6\u00b0)'
)
ax3.text(0.04, 0.97, txt, transform=ax3.transAxes,
         fontsize=10, family='monospace', verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.6))
ax3.set_title('Results', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Step 1:  \u0394x = {dx_p*1e3:.1f} mm   \u03c6\u2080 = {np.degrees(phi_0_s1):.1f}\u00b0')
print(f'Step 2:  \u03c6\u2080 = {np.degrees(phi_0_s2):.1f}\u00b0   (after shift correction)')
